# GenTwin: VAE-Driven Digital Twin for SWaT Cybersecurity

**Exploratory Analysis Notebook**

This notebook explores the GenTwin system for discovering hidden cybersecurity vulnerabilities in the SWaT water treatment testbed through:
- Variational Autoencoder (VAE) for synthetic attack generation
- Digital Twin physics simulation for impact validation
- Gap analysis to identify undetected high-risk scenarios

---

## 1. Environment Setup and Dependencies

In [ ]:
# Import required libraries
import sys
sys.path.append('..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# TensorFlow/Keras for VAE
import tensorflow as tf

# Scikit-learn for preprocessing and baselines
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.ensemble import IsolationForest
from sklearn.svm import OneClassSVM

# Scipy for physics simulation
from scipy.integrate import odeint

# Custom modules
from models.vae_model import create_vae
from digital_twin.swat_process import SWaTDigitalTwin
from baselines.detectors import create_baseline_detectors

# Set visualization style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

# Random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

print("All libraries imported successfully!")
print(f"TensorFlow version: {tf.__version__}")
print(f"NumPy version: {np.__version__}")
print(f"Pandas version: {pd.__version__}")

## 2. Data Loading and Exploration

Load the SWaT dataset and perform initial exploratory analysis.

In [ ]:
# Create sample SWaT-like data for demonstration
# In production, this would load the actual SWaT.A1_Dec 2015 dataset

n_samples = 10000
n_features = 51
np.random.seed(42)

# Generate normal operation data
data = np.random.randn(n_samples, n_features)
sensor_names = [f'Sensor_{i:02d}' for i in range(n_features)]

# Create DataFrame
df = pd.DataFrame(data, columns=sensor_names)
df['Timestamp'] = pd.date_range('2015-12-01', periods=n_samples, freq='1S')
df['Label'] = 'Normal'

# Display basic information
print("Dataset Shape:", df.shape)
print("\nFirst few rows:")
display(df.head())

print("\nBasic Statistics:")
display(df[sensor_names[:5]].describe())

## 3. VAE Model Testing

Test the VAE architecture for encoding/decoding sensor data.

In [ ]:
# Create VAE model
vae = create_vae(input_dim=51, latent_dim=32, beta=0.5)

print("VAE Model Created")
print("\nEncoder Summary:")
vae.encoder.summary()

print("\n" + "="*60)
print("Decoder Summary:")
vae.decoder.summary()

# Test encoding
sample_data = data[:100]
z_mean, z_log_var, z = vae.encode(sample_data)

print("\n" + "="*60)
print(f"Input shape: {sample_data.shape}")
print(f"Latent mean shape: {z_mean.shape}")
print(f"Latent sample shape: {z.shape}")

# Test reconstruction
reconstruction = vae.decode(z)
print(f"Reconstruction shape: {reconstruction.shape}")

# Calculate reconstruction error
mse = np.mean((sample_data - reconstruction.numpy())**2, axis=1)
print(f"\nMean reconstruction error: {np.mean(mse):.4f}")

## 4. Digital Twin Simulation

Test the Digital Twin process model and safety constraint checking.

In [ ]:
# Initialize Digital Twin
dt = SWaTDigitalTwin(dt=1.0)

print("Digital Twin Initialized")
print("\nInitial State:")
state = dt.get_system_state()
print(f"  Time: {state['time']}s")
print(f"  Safe: {state['is_safe']}")
print(f"  LIT101 (Tank Level): {state['state']['LIT101']:.2f} mm")
print(f"  AIT201 (Chlorine): {state['state']['AIT201']:.2f} mg/L")

# Simulate normal operation
print("\nSimulating 60 seconds of normal operation...")
for i in range(60):
    dt.step({})

state = dt.get_system_state()
print(f"  Time: {state['time']}s")
print(f"  Safe: {state['is_safe']}")
print(f"  LIT101: {state['state']['LIT101']:.2f} mm")

# Simulate attack (pump on with closed valve)
print("\nSimulating attack (pump deadheading)...")
for i in range(30):
    dt.step({'P101_status': 1.0, 'MV101_position': 0.0})

state = dt.get_system_state()
print(f"  Time: {state['time']}s")
print(f"  Safe: {state['is_safe']}")
print(f"  Violations: {len(state['violations'])}")

if state['violations']:
    print("\nViolation Details:")
    for v in state['violations']:
        print(f"  - {v['type']}: {v['sensor']} = {v['value']:.2f}")

## 5. Visualization: Sensor Time Series

Visualize sensor readings over time.

In [ ]:
# Plot first 5 sensors over time
fig, axes = plt.subplots(5, 1, figsize=(15, 10))

for i in range(5):
    sensor_name = sensor_names[i]
    axes[i].plot(df['Timestamp'][:1000], df[sensor_name][:1000], linewidth=0.5)
    axes[i].set_ylabel(sensor_name)
    axes[i].grid(True, alpha=0.3)

axes[0].set_title('Sensor Readings Over Time (First 1000 timesteps)', fontsize=14)
axes[-1].set_xlabel('Timestamp')

plt.tight_layout()
plt.show()

# Distribution visualization
fig, axes = plt.subplots(3, 3, figsize=(15, 10))
axes = axes.flatten()

for i in range(9):
    sensor_name = sensor_names[i]
    axes[i].hist(df[sensor_name], bins=50, alpha=0.7, edgecolor='black')
    axes[i].set_title(sensor_name)
    axes[i].set_xlabel('Value')
    axes[i].set_ylabel('Frequency')

plt.suptitle('Sensor Value Distributions', fontsize=16)
plt.tight_layout()
plt.show()

## 6. Summary and Next Steps

This notebook demonstrates the core components of the GenTwin system:

1. **VAE Model**: Successfully encodes 51-dimensional sensor data to 32-dimensional latent space
2. **Digital Twin**: Simulates water treatment process physics and detects safety violations
3. **Integration**: Both components work together to validate synthetic attacks

### Next Steps:
1. Run `data_pipeline.py` to preprocess actual SWaT dataset
2. Train VAE with `models/train_vae.py` for 100 epochs
3. Generate 1000 synthetic attacks with `attack_generator.py`
4. Analyze impacts with `digital_twin/impact_analyzer.py`
5. Launch dashboard with `streamlit run app.py`

### Expected Outcomes:
- Discover 15-25 hidden security gaps
- Achieve 85%+ detection with proposed enhancements
- 40% reduction in false positives vs baselines
- Identify 3-5 critical unmonitored components